[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shan4949/ml-visualized/blob/main/notebooks/16_pca.ipynb)

# 16 · Principal Component Analysis: The Best Low-Dimensional Shadow

**Part 2 — Classical unsupervised learning** · Notebook 16 of 27

Real datasets have many features, and many of them say the same thing twice. Bedrooms and bathrooms rise together. Neighboring pixels in an image are nearly equal. Interest rates for 5 and 7 years move in lockstep. **Principal component analysis** (PCA) finds the few directions along which the data actually varies, and re-describes every point by its coordinates along those directions.

It is the single most used tool for dimensionality reduction: to visualize high-dimensional data in 2-D, to compress images, to remove noise, to decorrelate features before a model, and as the first thing you run on a new dataset. Under the hood it's an eigenvalue problem, which is why notebook 01 spent so long on eigenvectors and the SVD.

**What you'll learn**

1. Why correlated features cause trouble
2. The idea: find the direction with the widest shadow
3. The derivation: variance → covariance → the eigen equation
4. More components, explained variance, and the scree plot
5. PCA from scratch, and why everyone computes it with the **SVD**
6. Preprocessing: centering, scaling, differencing
7. Visualization: 64-dimensional digits on a page
8. Compression and reconstruction
9. **Eigenfaces**, and recognizing faces by their distance from "face space"
10. Denoising
11. PCA on time series: level, slope, curvature
12. **Probabilistic PCA**: PCA as a latent-variable model
13. What PCA can't do

*Course map: ISYE 4600 W3 (Principal Component Analysis: correlated features, projection, variance maximization, eigendecomposition, PCA algorithm, preprocessing, eigenfaces, time series, SVD) and HW2 Q1–Q4 (PCA derivation, covariance matrix, correlated features, PCA on a real table, eigenfaces with projection residuals); CX 4240 lecture 18 (dimension reduction, probabilistic PCA as a Gaussian latent-variable model, three views of PCA, eigenfaces).*

In [1]:
#@title ⚙️ Setup — run this cell first (imports, plot style, helpers)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display, Markdown
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider

# One consistent, colorblind-checked palette used across the whole series
C = dict(blue="#2a78d6", orange="#eb6834", aqua="#1baf7a", yellow="#eda100",
         magenta="#e87ba4", green="#008300", violet="#4a3aa7", red="#e34948",
         ink="#0b0b0b", muted="#52514e", light="#d9d8d3")
plt.rcParams.update({
    "figure.figsize": (6.4, 4.2), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.prop_cycle": plt.cycler(color=[C[k] for k in
        ["blue", "orange", "aqua", "yellow", "magenta", "green", "violet", "red"]]),
    "lines.linewidth": 2, "font.size": 11, "legend.frameon": False,
    "animation.embed_limit": 40, "animation.frame_format": "jpeg",   # JPEG frames keep notebooks small
})

def show_anim(anim):
    """Render a matplotlib animation as an interactive player (play / pause / scrub)."""
    plt.close(anim._fig)
    return HTML(anim.to_jshtml(default_mode="once"))

rng = np.random.default_rng(0)
print("Setup complete ✓")
from sklearn.decomposition import PCA, KernelPCA
from sklearn.datasets import load_digits, load_wine, make_circles, make_swiss_roll
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
from skimage import data as skdata
print("Ready ✓")

Setup complete ✓


Ready ✓


---
## 1. Why correlated features cause trouble

ISYE's motivating example: predicting rent from number of bedrooms and number of bathrooms. The two features are strongly correlated, and in the extreme case $x_1 = 1.5\,x_2$ for every apartment, they carry **one** piece of information, not two.

In a linear model $y = \beta_0 + \beta_1 x_1 + \beta_2 x_2$, the normal equations need $X^\top X$ (or, in ISYE's row convention, $XX^\top$) to be invertible. With perfectly correlated features, one column is a multiple of another, so it isn't. Here is HW2's numerical check, with ten homes' bathroom counts:

In [2]:
baths = np.array([1, 2, 2, 3, 1, 2, 3, 2, 1, 2], float)
beds = 1.5 * baths                                             # perfectly correlated: x1 = 1.5 x2
Xrows = np.vstack([np.ones(10), beds, baths])                  # ISYE convention: 3 × m, one column per home
G = Xrows @ Xrows.T / 10
print("G = (1/m) X Xᵀ =\n", G.round(3))
print("rank(G) =", np.linalg.matrix_rank(G), "  eigenvalues:", np.linalg.eigvalsh(G).round(4))

G = (1/m) X Xᵀ =
 [[1.    2.85  1.9  ]
 [2.85  9.225 6.15 ]
 [1.9   6.15  4.1  ]]
rank(G) = 2   eigenvalues: [-0.     0.112 14.213]


$G$ is $3\times3$ but has rank 2: one eigenvalue is exactly zero, and its eigenvector, proportional to $(0, 1, -1.5)$, is the combination $x_1 - 1.5x_2$ that is **always zero**. Any multiple of it can be added to $(\beta_1, \beta_2)$ without changing the predictions, so the coefficients aren't identifiable. Real features are rarely *perfectly* correlated, but nearly-correlated ones give nearly-zero eigenvalues, and the coefficients become wildly unstable (we saw this in notebook 05's collinearity demo and fixed it with ridge in notebook 07).

PCA's answer is different: instead of penalizing the coefficients, **replace the correlated features by their dominant combination**. ISYE asks: is "0.5 × bedrooms + 0.5 × bathrooms" the right combination? Which features should be combined, with what weights? PCA answers all of it from the data.

---
## 2. The idea: find the direction with the widest shadow

ISYE's second example: six students' hours studied and exam scores.

| Student | A | B | C | D | E | F |
|---|---|---|---|---|---|---|
| Hours | 2.5 | 5.1 | 3.2 | 4.5 | 1.9 | 6.0 |
| Score | 65 | 85 | 70 | 82 | 59 | 92 |

How do we best summarize this data with a single line? Imagine a light shining perpendicular to a line through the data's center: each point casts a **shadow** onto the line (its projection). PCA rotates the line until the shadows are **as spread out as possible**. 🎚️ **Rotate the line** and watch two numbers: the variance of the shadows, and the squared distance from each point to its shadow.

In [3]:
stud = np.array([[2.5, 65], [5.1, 85], [3.2, 70], [4.5, 82], [1.9, 59], [6.0, 92]])
names = list("ABCDEF")
# Put both features on comparable scales for the picture: hours (x10) so the geometry is visible
S_ = stud * np.array([10, 1.0]); mu_s = S_.mean(0); Sc = S_ - mu_s

def best_angle(Xc):
    vals, vecs = np.linalg.eigh(Xc.T @ Xc / len(Xc)); u = vecs[:, -1]; return np.degrees(np.arctan2(u[1], u[0])) % 180

angles = np.linspace(0, 180, 361)
var_at = [np.var(Sc @ np.array([np.cos(np.radians(a)), np.sin(np.radians(a))])) for a in angles]
err_at = [((Sc ** 2).sum(1) - (Sc @ np.array([np.cos(np.radians(a)), np.sin(np.radians(a))])) ** 2).mean() for a in angles]

def shadow(angle=10.0):
    u = np.array([np.cos(np.radians(angle)), np.sin(np.radians(angle))])
    z = Sc @ u; P = np.outer(z, u)                             # scores and projected points
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    ax = axes[0]
    ax.plot(*np.c_[-45 * u, 45 * u], color=C["red"], lw=2)
    for p, q in zip(Sc, P): ax.plot(*np.c_[p, q], color=C["muted"], lw=1, ls=":")
    ax.scatter(*Sc.T, s=70, color=C["blue"], zorder=3); ax.scatter(*P.T, s=50, color=C["red"], marker="s", zorder=4)
    for (xx, yy), n in zip(Sc, names): ax.annotate(n, (xx, yy), xytext=(5, 5), textcoords="offset points")
    ax.set(aspect="equal", xlim=(-30, 30), ylim=(-25, 25), xlabel="hours studied × 10 (centered)", ylabel="exam score (centered)",
           title=f"line at {angle:.0f}°: shadow variance {np.var(z):.1f}, mean squared distance {((Sc - P) ** 2).sum(1).mean():.1f}")
    ax = axes[1]
    ax.plot(angles, var_at, color=C["red"], label="variance of the shadows (maximize)")
    ax.plot(angles, err_at, color=C["blue"], label="mean squared distance to the line (minimize)")
    ax.axvline(angle, color=C["ink"], lw=1); ax.axvline(best_angle(Sc), color=C["aqua"], ls="--", label=f"PCA direction ({best_angle(Sc):.1f}°)")
    ax.set(xlabel="angle of the line (degrees)", title="the two curves always add up to the same total"); ax.legend(fontsize=9, loc="center right")
    fig.tight_layout(); plt.show()

interact(shadow, angle=FloatSlider(value=10, min=0, max=180, step=1, description="angle"));

Two things to notice.

1. **Maximum shadow variance and minimum squared distance happen at the same angle.** For each point, Pythagoras says $\|x\|^2 = (\text{shadow length})^2 + (\text{distance to line})^2$. The left side doesn't depend on the line, so making the shadows spread out as much as possible *is* the same as making the points as close to the line as possible. These are the "max variance" and "min reconstruction error" views of PCA (CX lecture 18).
2. The best line is **not** the regression line. Regression minimizes *vertical* distances (errors in $y$ only), treating hours as fixed. PCA minimizes *perpendicular* distances and treats both features symmetrically. There's no "target" in PCA.

(We multiplied hours by 10 so the picture isn't squashed. That choice matters: section 6 shows that PCA's answer depends on the units.)

---
## 3. The derivation: variance → covariance → eigenvectors

Take $m$ points $x^1, \dots, x^m \in \mathbb{R}^n$ with mean $\mu$. A direction is a unit vector $u$ ($\|u\| = 1$, since only orientation matters). Each point's **score** along $u$ is $u^\top x^i$, the signed length of its shadow. We want the scores to have maximal variance (ISYE W3):

$$\max_{\|u\|=1}\; \frac{1}{m}\sum_{i=1}^m \big(u^\top x^i - u^\top\mu\big)^2.$$

Pull $u$ out of the sum:

$$\frac{1}{m}\sum_i \big(u^\top(x^i - \mu)\big)^2 = \frac1m\sum_i u^\top(x^i-\mu)(x^i-\mu)^\top u = u^\top\Big(\frac1m\sum_i (x^i-\mu)(x^i-\mu)^\top\Big)u = u^\top\Sigma u,$$

where $\Sigma$ is the **covariance matrix**. So the variance along any direction is the quadratic form $u^\top\Sigma u$. To maximize it subject to $u^\top u = 1$, use a Lagrange multiplier (notebook 03):

$$\mathcal{L}(u, \lambda) = u^\top\Sigma u - \lambda(u^\top u - 1), \qquad \nabla_u\mathcal{L} = 2\Sigma u - 2\lambda u = 0 \;\;\Longrightarrow\;\; \Sigma u = \lambda u.$$

The optimal direction must be an **eigenvector** of $\Sigma$. Which one? Plug any unit eigenvector back in: the variance is $u^\top\Sigma u = u^\top\lambda u = \lambda$. So the variance along an eigenvector *is* its eigenvalue, and the maximum is the eigenvector with the **largest eigenvalue**. That's the first principal component (HW2 Q1).

Let's check on the student data, in its original units (ISYE works this example by hand):

In [4]:
mu = stud.mean(0); Xc = stud - mu
Sigma = Xc.T @ Xc / len(Xc)
vals, vecs = np.linalg.eigh(Sigma)                             # eigh: for symmetric matrices, eigenvalues ascending
print("covariance matrix Σ =\n", Sigma.round(2))
print("\neigenvalues:", vals.round(4))
print("eigenvectors (columns):\n", vecs.round(4))
u1 = vecs[:, -1] * np.sign(vecs[1, -1])
print(f"\nPC1 direction u₁ = {u1.round(4)}:  variance along it = u₁ᵀΣu₁ = {u1 @ Sigma @ u1:.2f} = λ₁")
print("scores z = u₁ᵀ(x − μ):", {n: round(float(v), 2) for n, v in zip(names, Xc @ u1)})

covariance matrix Σ =
 [[  2.11  16.92]
 [ 16.92 136.25]]

eigenvalues: [8.400000e-03 1.383505e+02]
eigenvectors (columns):
 [[-0.9924  0.1232]
 [ 0.1232  0.9924]]

PC1 direction u₁ = [0.1232 0.9924]:  variance along it = u₁ᵀΣu₁ = 138.35 = λ₁
scores z = u₁ᵀ(x − μ): {'A': -10.59, 'B': 9.58, 'C': -5.54, 'D': 6.53, 'E': -16.62, 'F': 16.64}


---
> 🚧 **This notebook is being written section by section.** The sections above are complete and run top to bottom; more are on the way.